# A 策略：固定 25 檔的權重比較
沿用既有持續 kernel 的資料與帳戶引擎，不重啟。固定 90% 投資、每五交易日調整與原成本；比較等權、溫和動能加權、逆波動配置。探索性實驗，依賴原基準 Notebook 的已載入狀態。

In [ ]:
# Locate this competition repository; do not restart an existing kernel.
import sys as _repo_sys, importlib.util as _repo_importlib
from pathlib import Path as _PlainPath
_candidates = [_PlainPath.cwd(), *_PlainPath.cwd().parents]
_repo_root = next((p for p in _candidates if (p / '99_維護與備份' / '工具' / '研究路徑.py').exists()), None)
if _repo_root is None:
    raise RuntimeError('Open Jupyter from this repository root or its 04_Notebook directory.')
if 'research_layout' not in _repo_sys.modules:
    _spec = _repo_importlib.spec_from_file_location('research_layout', _repo_root / '99_維護與備份' / '工具' / '研究路徑.py')
    _mod = _repo_importlib.module_from_spec(_spec)
    _repo_sys.modules['research_layout'] = _mod
    _spec.loader.exec_module(_mod)
from research_layout import ResearchPath as Path
%whos
import inspect
print('Simulator source:')
print(inspect.getsource(simulate))
print('Constants:', {k:globals().get(k) for k in ['COUNT','INVESTED','HORIZON','REBALANCE','LOT','INITIAL','FEE','TAX']})
print('Signal return shape:', signal_returns.shape)
print('Kernel environment:', __import__('sys').executable)

## 集中參數與可比性
三方案每次都由 A 訊號取同一 25 檔、投資 90%、每五交易日重選及調整。等權每檔 3.6%；溫和動能按入選排名給 1.25 至 0.75 的線性倍率，正規化後為 4.5% 至 2.7%。逆波動用前 20 日已調整日報酬的標準差，先算逆波動權重，再與等權各混合一半，限制每檔目標 1.8% 至 5.4%、合計 90%。這些是探索性固定參數，不是文獻已證明最佳的比例。

目標限制不等於每天實際權重上限：成交均價、整張與價格漂移仍需逐日檢查。沿用原 10%／台積電 25% 上限及原引擎五日漂移計數，另獨立計當日超限天數。保留原始 650 日期軸、共同可判定窗口比較；不清倉計期末總資產，費用同原 A。

比較母體修正：主比較只要求本次三配置共同可判定，不因無關的 REF／B／C 資料缺漏排除；另保留原六方案共同 375 日期的敏感性表。2025 同季起點的 A 可判定，原本因 REF 缺漏排除，不應稱 A 本身缺資料。

In [ ]:
import numpy as np,inspect,datetime
weight_experiment_labels={'equal':'A 等權','mild':'A 溫和動能加權','invvol':'A 等權＋逆波動'}
weight_experiment_audit={k:[] for k in weight_experiment_labels}
def weight_bounded_normalize(raw,total=.9,lower=.018,upper=.054):
    raw=np.asarray(raw,dtype=float)
    assert np.all(np.isfinite(raw)&(raw>0)) and len(raw)*lower<=total<=len(raw)*upper
    lo,hi=0.,total/min(raw)
    for _ in range(80):
        mid=(lo+hi)/2
        if np.clip(mid*raw,lower,upper).sum()<total:lo=mid
        else:hi=mid
    w=np.clip((lo+hi)/2*raw,lower,upper)
    assert abs(w.sum()-total)<1e-10
    return w
def weight_experiment_allocation(t,selected,allocation):
    if allocation=='equal':w=np.full(COUNT,INVESTED/COUNT)
    elif allocation=='mild':
        multiplier=np.linspace(1.25,.75,COUNT)
        w=INVESTED*multiplier/multiplier.sum()
    elif allocation=='invvol':
        vol=np.std(signal_returns[t-20:t,selected],axis=0)
        if np.any(~np.isfinite(vol)) or np.any(vol<=0):raise ValueError('Invalid allocation volatility')
        inverse=1/vol
        raw=.5*np.full(COUNT,INVESTED/COUNT)+.5*INVESTED*inverse/inverse.sum()
        w=weight_bounded_normalize(raw)
    else:raise ValueError('Unknown allocation')
    assert np.all(np.isfinite(w)&(w>0)) and abs(w.sum()-INVESTED)<1e-10
    assert np.max(w)<=.054+1e-10
    weight_experiment_audit[allocation].append((int(t),tuple(int(x) for x in selected),w.copy()))
    return w
# Reuse exact accounting body. Only the allocation line and additional audit change.
weight_simulator_source=inspect.getsource(simulate)
weight_simulator_source=weight_simulator_source.replace('def simulate(start,strategy,horizon=HORIZON):','def simulate_weights(start,allocation,horizon=HORIZON):\n    strategy="A"')
original_allocation='target[selected]=(np.floor((INVESTED/COUNT)*previous_nav/close[t-1,selected]/LOT)*LOT).astype(np.int64)'
assert weight_simulator_source.count(original_allocation)==1
weight_simulator_source=weight_simulator_source.replace(original_allocation,'target[selected]=(np.floor(weight_experiment_allocation(t,selected,allocation)*previous_nav/close[t-1,selected]/LOT)*LOT).astype(np.int64)')
weight_simulator_source=weight_simulator_source.replace('violations=0; min_cash=', 'strict_cap_days=0; max_actual_weight=0.; min_held=COUNT; max_held=0; violations=0; min_cash=')
weight_simulator_source=weight_simulator_source.replace('cap_age=np.where(weights>caps+1e-12,cap_age+1,0)', 'strict_cap_days += int(np.any(weights>caps+1e-12))\n        max_actual_weight=max(max_actual_weight,float(weights.max()))\n        min_held=min(min_held,int(np.count_nonzero(shares)))\n        max_held=max(max_held,int(np.count_nonzero(shares)))\n        cap_age=np.where(weights>caps+1e-12,cap_age+1,0)')
weight_simulator_source=weight_simulator_source.replace("'path':vals}","'path':vals,'strict_cap_days':strict_cap_days,'max_actual_weight':max_actual_weight,'min_held':min_held,'max_held':max_held}")
exec(weight_simulator_source)
weight_experiment_results={k:[simulate_weights(int(s),k) for s in window_starts] for k in weight_experiment_labels}
# Equal-weight experiment must reproduce original A, including invalid diagnostic paths.
for old,new in zip(backtest_results['A'],weight_experiment_results['equal']):
    assert old['invalid']==new['invalid'] and old['violations']==new['violations']
    for field in ['return','mdd','cost','turnover']:assert np.isclose(old[field],new[field],rtol=1e-12,atol=1e-12)
    assert np.allclose(old['path'],new['path'],rtol=1e-12,atol=1e-6)
for k in ['mild','invvol']:
    assert len(weight_experiment_audit[k])==len(weight_experiment_audit['equal'])
    assert all(a[:2]==b[:2] for a,b in zip(weight_experiment_audit['equal'],weight_experiment_audit[k]))
weight_experiment_valid=np.ones(len(window_starts),dtype=bool)
for results in weight_experiment_results.values():weight_experiment_valid &= np.array([not x['invalid'] for x in results])
weight_experiment_metrics={k:{field:np.array([x[field] for x in results]) for field in ['return','mdd','cost','turnover','violations','strict_cap_days','max_actual_weight','min_held','max_held']} for k,results in weight_experiment_results.items()}
print('Original A reproduced, selections identical across all allocation events.')
print('Windows:',len(window_starts),'common valid:',int(weight_experiment_valid.sum()),'original-six shared:',int(np.sum(common_valid&weight_experiment_valid)),'restored A-valid:',int(np.sum(weight_experiment_valid&~common_valid)))
print('Valid-window violations:',{k:int(v['violations'][weight_experiment_valid].sum()) for k,v in weight_experiment_metrics.items()})
print('Strict cap days:',{k:int(v['strict_cap_days'][weight_experiment_valid].sum()) for k,v in weight_experiment_metrics.items()})
print('Max actual weight / held range:',{k:(float(v['max_actual_weight'][weight_experiment_valid].max()),int(v['min_held'][weight_experiment_valid].min()),int(v['max_held'][weight_experiment_valid].max())) for k,v in weight_experiment_metrics.items()})


In [ ]:
from IPython.display import Markdown,display
weight_experiment_masks={'全部':weight_experiment_valid,'2024–2025':weight_experiment_valid&np.array([dates[s+HORIZON-1]<datetime.date(2026,1,1) for s in window_starts]),'2026':weight_experiment_valid&np.array([dates[s]>=datetime.date(2026,1,1) for s in window_starts]),'原六方案共同日期敏感性':weight_experiment_valid&common_valid}
weight_experiment_tables={}
weight_experiment_ci={}
for period,mask in weight_experiment_masks.items():
    lines=['|配置|窗口數|報酬中位數|報酬P10|MDD中位數|平均成本|平均雙邊換手|勝過等權|配對平均超額|','|---|---:|---:|---:|---:|---:|---:|---:|---:|']
    base=weight_experiment_metrics['equal']['return']
    for k,label in weight_experiment_labels.items():
        v=weight_experiment_metrics[k]; rr=v['return'][mask]; excess=v['return']-base
        vals=[np.median(rr),np.quantile(rr,.1),np.median(v['mdd'][mask]),np.mean(v['cost'][mask]),np.mean(v['turnover'][mask]),np.mean(excess[mask]>1e-12),np.mean(excess[mask])]
        lines.append('|'+label+'|'+str(len(rr))+'|'+'|'.join(f'{x:.2%}' for x in vals)+'|')
        if k!='equal':
            positions=np.arange(len(window_starts)) if period in ['全部','原六方案共同日期敏感性'] else np.flatnonzero(np.array([(dates[s+HORIZON-1]<datetime.date(2026,1,1)) if period=='2024–2025' else (dates[s]>=datetime.date(2026,1,1)) for s in window_starts]))
            series=np.where(mask,excess,np.nan)[positions]
            weight_experiment_ci[(period,k)]={b:block_ci(series,b).tolist() for b in [24,48]}
    weight_experiment_tables[period]='\\n'.join(lines).replace('\\n','\n')
    display(Markdown('### '+period+'\n\n'+weight_experiment_tables[period]))
for (period,k),cis in weight_experiment_ci.items():print(period,k,'paired-mean excess CI:',{b:[f'{x:.2%}' for x in v] for b,v in cis.items()})
# Exactly the historical competition anchors, not a broad month filter.
weight_experiment_anchor_lines=['|年份／起點|有效狀態|等權|溫和動能|等權＋逆波動|','|---|---|---:|---:|---:|']
for year,anchor in [(2024,datetime.date(2024,10,29)),(2025,datetime.date(2025,10,28))]:
    idx=np.flatnonzero(np.array([dates[s]==anchor for s in window_starts]))
    assert len(idx)==1
    i=int(idx[0]);valid=bool(weight_experiment_valid[i])
    weight_experiment_anchor_lines.append('|'+str(year)+'／'+str(anchor)+'|'+('共同有效' if valid else '排除，暫算')+'|'+'|'.join(f"{weight_experiment_metrics[k]['return'][i]:.2%}" for k in weight_experiment_labels)+'|')
weight_experiment_anchor_table='\n'.join(weight_experiment_anchor_lines)
display(Markdown('### 各年十月最後星期二：完整 24 日\n\n'+weight_experiment_anchor_table))


In [ ]:
import json
weight_anchor_diagnostics=[]
diagnostic_source=weight_simulator_source.replace('def simulate_weights(start,allocation,horizon=HORIZON):','def weight_anchor_diagnostic(start,allocation,horizon=HORIZON):\n    reasons=[]')
diagnostic_source=diagnostic_source.replace('invalid |= bool(np.any(bad_action[t] & ((shares+receivable_shares)>0)))','affected=np.flatnonzero(bad_action[t] & ((shares+receivable_shares)>0))\n        if len(affected):reasons.append({"date":str(dates[t]),"reason":"unsupported corporate action or unexplained drop", "stock_ids":[stock_ids[j] for j in affected]})\n        invalid |= bool(len(affected))')
diagnostic_source=diagnostic_source.replace('gross_traded += traded; fees += cost; invalid |= missing','if missing:reasons.append({"date":str(dates[t]),"reason":"unavailable execution price"})\n            gross_traded += traded; fees += cost; invalid |= missing')
diagnostic_source=diagnostic_source.replace("'max_held':max_held}","'max_held':max_held,'reasons':reasons}")
# Raw string replace values must produce actual indentation/newlines in the generated function.
diagnostic_source=diagnostic_source.replace('\\n','\n')
exec(diagnostic_source)
for anchor in [datetime.date(2024,10,29),datetime.date(2025,10,28)]:
    i=int(np.flatnonzero(np.array([dates[s]==anchor for s in window_starts]))[0])
    before_audit=len(weight_experiment_audit['equal'])
    result=weight_anchor_diagnostic(int(window_starts[i]),'equal')
    del weight_experiment_audit['equal'][before_audit:]
    item={'anchor':str(anchor),'original_invalid':{k:bool(v[i]['invalid']) for k,v in backtest_results.items()},'new_allocation_invalid':{k:bool(v[i]['invalid']) for k,v in weight_experiment_results.items()},'A_reasons':result['reasons']}
    weight_anchor_diagnostics.append(item)
print(json.dumps(weight_anchor_diagnostics,ensure_ascii=False,default=str))


## 判讀與限制
溫和動能加權在 447 共同有效窗口的配對平均超額約 0.11 個百分點，但 24／48 位置區塊重抽樣區間均包含零，2026 配對平均超額轉負；保留研究候選，不取代等權。逆波動混合全期配對平均超額約 -0.17 個百分點，2026 為 -0.28 個百分點且本次區塊區間在零以下，暫不優先。不是所有逆波動配置皆無效的結論，也不是多重比較校正後的確認性檢定。

主母體只要求本次三配置共同有效，不因 REF／B／C 缺漏排除；原六策略 375 日期另作敏感性檢查，結論無實質改变。不同母體不可直接與舊六方案總表排名。2024 同季窗口因 11/6 無成交價排除；2025 同季 A 有效，原排除來自 REF。

本次精確複製原 A 等權在 650 窗口的報酬、成本、換手、回撤及資產路徑；三配置所有調倉時點選股一致。447 有效窗口持股均為 25 檔，現金／持股數及原漂移檢查違規為零，當日嚴格單股超限亦為零。這不代表完整官方合規，Active Share、成交量限制、市場衝擊、文件提交及整批作廢均未驗證。

參數與研究方向已參考過既有資料，2026 只是時間穩定性檢查，不是封存的正式樣本外。窗口重疊、203／650 排除、原 150 檔歷史名單偏誤及公司行動缺漏仍限制結論。Notebook 依賴既有 kernel 的原始物件；未重啟 Run All，也不能稱為獨立可重跑。原 Notebook、資料與 DB 未更動。


In [ ]:
weight_report_directory='C:\\Users\\weiha\\Desktop\\文件\\03_課程資料\\115-1\\資料探勘與應用\\B_Final Project (0920~1221)\\競賽資料'
from research_layout import ResearchPath as Path
import json
weight_report_text='# A 策略：25 檔權重配置實驗\n\n日期：2026-10-14。所有計算在既有 aicup-backtest-20261013 kernel；未重啟、未重新查詢 DB、未提交交易。\n\n'
weight_report_text+='## 固定實驗設定\n\n同一 A 20／60 日訊號、相同入選 25 檔，目標投資 90%、每五交易日調整、每次使用前日資料；整千股、當日均價成交。買賣費各 0.1425%、賣出稅 0.3%，期末不假設清倉。\n\n'
weight_report_text+='- 等權：每檔 3.6%。\n- 溫和動能：按入選排名給 1.25→0.75 線性倍率，每檔 4.5%→2.7%。\n- 等權＋逆波動：前 20 日調整日報酬標準差的逆數正規化，与等權各混合一半；每檔目標限制 1.8%–5.4%，合計 90%。用尺度二分及截斷保持總和與界限。\n\n以上是先定好的探索參數，没有搜尋最佳比例；逆波動配置仍是相同動能股票，不是獨立策略家族。目標上限不等於實際每日權重。\n\n'
weight_report_text+='## 三配置共同可判定結果\n\n650 個窗口中 447 個有效、203 個排除。主比較不因無關的 REF／B／C 缺漏排除；原六方案共有的 375 日期另列。全部 447 包含 16 個跨越年份邊界的窗口，早期／2026 分表排除跨界，因此分表窗口數之和不同。不同母體不可直接拿舊六策略表排名。\n\n'
for title,table in weight_experiment_tables.items():weight_report_text+='### '+title+'\n\n'+table+'\n\n'
weight_report_text+='勝過等權是同日期費用後報酬較高的比例，不是正報酬率或競賽奪冠率；等權對自己的 0% 沒有投資意義。P10 是報酬第十百分位；MDD 是負值的最大回撤。配對超額的單位是百分點。\n\n'
weight_report_text+='## 配對平均超額的區塊重抽樣\n\n沿用原種子 SEED＋區塊長度、每組 2,000 次；完整窗口位置軸保留缺漏，24／48 位置區塊，不把窗口當獨立。95% 區間，不做多重比較校正。\n\n|區間|配置|24 位置區塊|48 位置區塊|\n|---|---|---|---|\n'
for (period,k),cis in weight_experiment_ci.items():
    weight_report_text+='|'+period+'|'+weight_experiment_labels[k]+'|'+'|'.join(f'{cis[b][0]*100:.3f} 至 {cis[b][1]*100:.3f} 個百分點' for b in [24,48])+'|\n'
weight_report_text+='\n## 與競賽同季的精確起點\n\n'
weight_report_text+=weight_experiment_anchor_table.replace('|2024／2024-10-29|排除，暫算|6.93%|8.04%|6.33%|','|2024／2024-10-29|不可判定|—|—|—|')+'\n\n'
weight_report_text+='2024 A 在 11/6 缺委託成交價格，不將暫算值當績效；2025 三配置有效，原六方案排除原因是 REF，不是 A。只有一个有效的同季起點，不足判定同季配置優劣。\n\n'
weight_report_text+='## 判斷與下一步\n\n溫和動能加權全期平均超額約 +0.11 個百分點、勝過等權 62.42%，但兩種全期區塊區間都含零；2026 平均超額約 -0.12 個百分點。中位數較高与平均配對較差可以並存，不能只挑中位數作結論。保留研究候選，沒有證據要求立即取代等權。\n\n'
weight_report_text+='逆波動混合的全期平均超額約 -0.17 個百分點；2026 約 -0.28 個百分點且此輪兩個區塊區間低於零。部分回撤／P10 改善沒有轉成較高的費用後報酬，暫不優先；不能外推到其他逆波動參數。原 375 日期敏感性亦未顯示明確優勢。\n\n'
weight_report_text+='下一輪優先研究調整頻率與交易門檻，確認保留動能優勢是否需要每五日全量調整；之後才研究可重複的公告配置。尚未開始下一輪回測，原 A 等權仍保留作對照，不宣稱它已是最適配置。\n\n'
weight_report_text+='## 驗證、保存與限制\n\n650 窗口等權與原 A 報酬、回撤、成本、換手及逐日資產路徑一致；三配置所有調倉選股一致。447 有效窗口均持有 25 檔，原現金／檔數／漂移計數違規为零，當日嚴格單股超限亦为零；最大觀察單股權重約等權 5.49%、溫和動能 6.23%、逆波動混合 6.27%。未完整驗證 Active Share、成交量、市場衝擊、文件提交或整批作廢。\n\n'
weight_report_text+='沿用原資料與公司行動缺漏排除規則；150 檔名單回推有存活偏誤。參數受既有研究啟發，2026 不是封存樣本外。Notebook 保存輸出，但依賴既有 kernel 物件，未獲清除狀態授權所以沒有 Restart and Run All。原動能／公告 Notebook 未改動。\n\n'
weight_report_text+='關鍵物件：weight_experiment_results、weight_experiment_metrics、weight_experiment_valid、weight_experiment_ci、weight_experiment_audit、weight_anchor_diagnostics。大型資料留在 kernel；輸出不保存行情全集或 DB 密碼。\n\n'
weight_report_text+='[計算 Notebook](動能基準研究_25檔權重比較_2026-10-14.ipynb)。\n'
weight_report_path=Path(weight_report_directory)/'動能基準研究_25檔權重比較_2026-10-14.md'
weight_report_path.write_text(weight_report_text,encoding='utf-8')
print('Saved',weight_report_path.name)
